# Train a model, save it, use it

We train a straight-line regression once, save it to a file, and then use the saved model in an app.

The data here is **made up**, to show the steps. A real business would use its own sales records.

In [5]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score


## Made-up data

Two hundred days. Hotter days with less rain sell more ice cream, with some random noise.

In [2]:

np.random.seed(42)
n = 200

# Generate weather features
temperature = np.random.uniform(50, 100, n)
rain_probability = np.random.uniform(0, 100, n)

# Generate synthetic ice cream sales
sales = (
    5 * temperature
    - 1.2 * rain_probability
    - 100
    + np.random.normal(0, 25, n)
)

# Create DataFrame
df = pd.DataFrame({
    "temperature": temperature,
    "rain_probability": rain_probability,
    "sales": sales.clip(0)
})

df.head()


,temperature,rain_probability,sales
0,68.727006,64.203165,199.228202
1,97.535715,8.413996,378.106877
2,86.599697,16.162871,330.651864
3,79.932924,89.855419,184.081450
4,57.800932,60.642906,124.337332


In [4]:
import plotly.express as px

px.scatter(
    df,
    x="temperature",
    y="sales",
    title="Ice Cream Sales vs. Temperature"
).show()

px.scatter(
    df,
    x="rain_probability",
    y="sales",
    title="Ice Cream Sales vs. Rain Probability"
).show()

## Fit the line

In [6]:

# Define features and target
X = df[["temperature", "rain_probability"]]
y = df["sales"]

# Train the linear regression model
model = LinearRegression()
model.fit(X, y)

# Display model coefficients
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)


Intercept: -98.19512142033048
Coefficients: [ 5.00491005 -1.24507779]


In [8]:

import plotly.graph_objects as go

# Create a grid of temperature and rain probability values
temp_grid = np.linspace(50, 100, 30)
rain_grid = np.linspace(0, 100, 30)

T, P = np.meshgrid(temp_grid, rain_grid)

# Predict sales across the grid
grid_df = pd.DataFrame({
    "temperature": T.ravel(),
    "rain_probability": P.ravel()
})

Z = model.predict(grid_df).reshape(T.shape)

# Plot actual sales data
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=df["temperature"],
    y=df["rain_probability"],
    z=df["sales"],
    mode="markers",
    name="Actual Sales",
    marker=dict(size=3, opacity=0.6)
))

# Add fitted regression plane
fig.add_trace(go.Surface(
    x=T,
    y=P,
    z=Z,
    opacity=0.6,
    name="Regression Plane",
    showscale=False
))

fig.update_layout(
    title="Multiple Linear Regression: Ice Cream Sales",
    scene=dict(
        xaxis_title="Temperature (°F)",
        yaxis_title="Rain Probability (%)",
        zaxis_title="Sales (cones)"
    ),
    width=850,
    height=650
)

fig.show()


/Users/lun/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/lun/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/lun/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [9]:

# Predict sales for a new day
new_day = pd.DataFrame({
    "temperature": [85],
    "rain_probability": [30]
})

prediction = model.predict(new_day)[0]

print(f"Predicted sales: {prediction:.0f} cones")


Predicted sales: 290 cones


## Save it

`pickle` writes the trained model to a file. We train once and use the file as many times as we like.

In [10]:

import pickle

# Save the trained model
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully!")


Model saved successfully!


## Load it back and check that it works

This is exactly what the app will do.

In [11]:
with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print(round(loaded_model.predict(new_day)[0]))   # the same number as before

290
